In [2]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("../")

import numpy as np
from src.predictors.chronos import Chronos
from transformers import PreTrainedModel


/Users/louisskowronek/Documents/thesis/master-thesis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-06-30 14:26:08,758 - INFO - config.py - PyTorch version 2.5.1 available.


# get number of trainable params for last layer fine tuning

In [ ]:
def print_trainable_params(model: PreTrainedModel) -> None:
    """Computes fraction of trainable params for a PreTrainedModel"""

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    fraction_trainable_params = trainable_params / total_params

    fraction_trainable_params = np.round(fraction_trainable_params * 100, 2)

    print(f"trainable params: {trainable_params} || all params: {total_params} || trainable%: {fraction_trainable_params}")

In [19]:
for size in ["tiny", "mini", "small", "base"]:
    chronos = Chronos(**{
                "pretrained_model_name_or_path": f"amazon/chronos-bolt-{size}",
                "device_map": "mps",
            })
    
    # Freeze all parameters
    for param in chronos.pipeline.inner_model.parameters():
        param.requires_grad = False

    for param in chronos.pipeline.inner_model.output_patch_embedding.output_layer.parameters():
        param.requires_grad = True
    print("Model:", size)
    print_trainable_params(chronos.pipeline.inner_model)

2025-06-30 14:34:28,829 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-bolt-tiny
2025-06-30 14:34:28,831 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-bolt-tiny
2025-06-30 14:34:29,683 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-bolt-mini
2025-06-30 14:34:29,684 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-bolt-mini


Model: tiny
trainable params: 590400 || all params: 8652672 || trainable%: 6.82


2025-06-30 14:34:30,813 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-bolt-small
2025-06-30 14:34:30,813 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-bolt-small


Model: mini
trainable params: 885312 || all params: 21236096 || trainable%: 4.17


2025-06-30 14:34:31,911 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-bolt-base
2025-06-30 14:34:31,912 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-bolt-base


Model: small
trainable params: 1180224 || all params: 47718016 || trainable%: 2.47
Model: base
trainable params: 1770048 || all params: 205292928 || trainable%: 0.86


In [18]:
for size in ["tiny", "mini", "small", "base", "large"]:
    chronos = Chronos(**{
                "pretrained_model_name_or_path": f"amazon/chronos-t5-{size}",
                "device_map": "mps",
            })
    
    # Freeze all parameters
    for param in chronos.pipeline.inner_model.parameters():
        param.requires_grad = False

    for param in chronos.pipeline.inner_model.lm_head.parameters():
        param.requires_grad = True
    print("Model:", size)
    print_trainable_params(chronos.pipeline.inner_model)

2025-06-30 14:34:16,956 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-t5-tiny
2025-06-30 14:34:16,956 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-t5-tiny
2025-06-30 14:34:17,899 - INFO - chronos.py - Context length detected of: 2048. Adapt context length to maximum of 512
2025-06-30 14:34:17,901 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-t5-mini
2025-06-30 14:34:17,901 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-t5-mini


Model: tiny
trainable params: 1048576 || all params: 8394496 || trainable%: 12.49


2025-06-30 14:34:18,660 - INFO - chronos.py - Context length detected of: 2048. Adapt context length to maximum of 512
2025-06-30 14:34:18,661 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-t5-small
2025-06-30 14:34:18,662 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-t5-small


Model: mini
trainable params: 1572864 || all params: 20456192 || trainable%: 7.69


2025-06-30 14:34:19,628 - INFO - chronos.py - Context length detected of: 2048. Adapt context length to maximum of 512
2025-06-30 14:34:19,630 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-t5-base
2025-06-30 14:34:19,630 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-t5-base


Model: small
trainable params: 2097152 || all params: 46154240 || trainable%: 4.54


2025-06-30 14:34:21,133 - INFO - chronos.py - Context length detected of: 2048. Adapt context length to maximum of 512
2025-06-30 14:34:21,136 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-t5-large
2025-06-30 14:34:21,136 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-t5-large


Model: base
trainable params: 3145728 || all params: 201374976 || trainable%: 1.56


2025-06-30 14:34:28,791 - INFO - chronos.py - Context length detected of: 2048. Adapt context length to maximum of 512


Model: large
trainable params: 4194304 || all params: 708963328 || trainable%: 0.59


# get number of trainable params for lora fine tuning

In [16]:
from peft import get_peft_model, LoraConfig, TaskType

In [17]:
for size in ["tiny", "mini", "small", "base", "large"]:
    chronos = Chronos(**{
                "pretrained_model_name_or_path": f"amazon/chronos-t5-{size}",
                "device_map": "mps",
            })


    lora_config = LoraConfig(
                    r=8,
                    lora_alpha=8,
                    target_modules=["q", "v", "k"],
                    lora_dropout=0.0,
                    task_type=None,
                )
    
    lora_model = get_peft_model(chronos.pipeline.inner_model, lora_config)
    print("Model:", size)
    lora_model.print_trainable_parameters()

2025-06-30 14:33:26,084 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-t5-tiny
2025-06-30 14:33:26,085 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-t5-tiny
2025-06-30 14:33:27,454 - INFO - chronos.py - Context length detected of: 2048. Adapt context length to maximum of 512
2025-06-30 14:33:27,499 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-t5-mini
2025-06-30 14:33:27,500 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-t5-mini


Model: tiny
trainable params: 147,456 || all params: 8,541,952 || trainable%: 1.7263


2025-06-30 14:33:28,582 - INFO - chronos.py - Context length detected of: 2048. Adapt context length to maximum of 512
2025-06-30 14:33:28,629 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-t5-small
2025-06-30 14:33:28,629 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-t5-small


Model: mini
trainable params: 258,048 || all params: 20,714,240 || trainable%: 1.2458


2025-06-30 14:33:29,698 - INFO - chronos.py - Context length detected of: 2048. Adapt context length to maximum of 512
2025-06-30 14:33:29,752 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-t5-base
2025-06-30 14:33:29,753 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-t5-base


Model: small
trainable params: 442,368 || all params: 46,596,608 || trainable%: 0.9494


2025-06-30 14:33:32,364 - INFO - chronos.py - Context length detected of: 2048. Adapt context length to maximum of 512
2025-06-30 14:33:32,467 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-t5-large
2025-06-30 14:33:32,467 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-t5-large


Model: base
trainable params: 1,327,104 || all params: 202,702,080 || trainable%: 0.6547


2025-06-30 14:33:39,422 - INFO - chronos.py - Context length detected of: 2048. Adapt context length to maximum of 512


Model: large
trainable params: 3,538,944 || all params: 712,502,272 || trainable%: 0.4967


In [ ]:
for size in ["tiny", "mini", "small", "base"]:
    chronos = Chronos(**{
                "pretrained_model_name_or_path": f"amazon/chronos-bolt-{size}",
                "device_map": "cuda",
            })


    lora_config = LoraConfig(
                    r=8,
                    lora_alpha=8,
                    target_modules=["q", "v", "k"],
                    lora_dropout=0.0,
                    bias="none",
                    task_type=TaskType.SEQ_2_SEQ_LM,
                )
    lora_model = get_peft_model(chronos.pipeline.inner_model, lora_config)
    print("Model:", size)
    lora_model.print_trainable_parameters()

2025-06-17 15:02:57,308 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-bolt-tiny
2025-06-17 15:02:57,309 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-bolt-tiny
2025-06-17 15:02:57,894 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-bolt-mini
2025-06-17 15:02:57,895 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-bolt-mini


Model: tiny
trainable params: 147,456 || all params: 8,800,128 || trainable%: 1.6756


2025-06-17 15:02:58,526 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-bolt-small
2025-06-17 15:02:58,526 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-bolt-small


Model: mini
trainable params: 258,048 || all params: 21,494,144 || trainable%: 1.2006


2025-06-17 15:02:59,154 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-bolt-base
2025-06-17 15:02:59,155 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-bolt-base


Model: small
trainable params: 442,368 || all params: 48,160,384 || trainable%: 0.9185
Model: base
trainable params: 1,327,104 || all params: 206,620,032 || trainable%: 0.6423
